### Install Libraries

In [ ]:
!pip install fastapi uvicorn nest-asyncio python-multipart transformers torch pillow omegaconf

### Create Config File

In [ ]:
yaml_config = """
  model_path: "Salesforce/blip-image-captioning-base"
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)
print("Đã tạo file config.yaml thành công!")

TEST MODEL

In [ ]:
import torch
import requests
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

print("Đang tải mô hình BLIP, vui lòng đợi...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

img_url = 'https://thumbs.dreamstime.com/b/portrait-man-tiger-red-haired-white-shirt-lies-next-to-bengal-strokes-striped-belly-164248427.jpg'

print("Kiểm tra input\n")
try:
  response = requests.get(img_url, stream=True, timeout=10)
  response.raise_for_status()
  raw_image = Image.open(response.raw).convert('RGB')
  print("Đầu vào hợp lệ! Đã tải xong ảnh mẫu")

  inputs = processor(raw_image, return_tensors="pt")

  print("Mô hình đang suy luận")
  with torch.no_grad():
      out = model.generate(**inputs)

  predicted_caption = processor.decode(out[0], skip_special_tokens=True)

  print(f"Ảnh mẫu: {img_url}")
  print(f"Câu mô tả do AI dự đoán: '{predicted_caption}'")
except requests.exceptions.RequestException as e:
    print(f"LỖI MẠNG/URL: Không thể tải dữ liệu từ đường link này. (Chi tiết: {e})")
except UnidentifiedImageError:
    print("LỖI DỮ LIỆU: File tải về không phải là hình ảnh hợp lệ hoặc file đã bị hỏng!")
except Exception as e:
    print(f"LỖI HỆ THỐNG: Có lỗi bất ngờ xảy ra trong quá trình xử lý: {e}")

### Build Model

In [6]:
import torch
from omegaconf import OmegaConf
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

class ImageCaptioningModel:
    def __init__(self, config_path):
        # Đọc đường dẫn mô hình từ file config.yaml
        self.config = OmegaConf.load(config_path)

        # kiểm tra xem máy đang có GPU hay chỉ có CPU
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Hệ thống đang sử dụng thiết bị tính toán: {self.device.upper()}")

        # Tải Processor và Model
        print("Đang tải mô hình từ Hugging Face vào bộ nhớ...")
        self.processor = BlipProcessor.from_pretrained(self.config.model_path)
        self.model = BlipForConditionalGeneration.from_pretrained(self.config.model_path).to(self.device)
        print("Tải mô hình thành công và đã sẵn sàng!")

    def predict(self, image: Image.Image):
        inputs = self.processor(image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs)
        caption = self.processor.decode(out[0], skip_special_tokens=True)
        return caption

### Initialize Model

In [ ]:
# Khởi tạo một instance từ Class
classifier = ImageCaptioningModel("./config.yaml")

### Initialize API

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image, UnidentifiedImageError
import io
import time
import threading
import uvicorn
import nest_asyncio

# Khởi tạo mô hình FastAPI
app = FastAPI(title="Image Captioning API")

# Cấu hình CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# Định nghĩa các Endpoint API
@app.get("/")
def read_root():
    """Thông tin giới thiệu hệ thống"""
    return {
        "name": "BLIP Image Captioning API",
        "version": "1.0.0",
        "description": "API phục vụ mô hình BLIP sinh mô tả hình ảnh."
    }

@app.get("/health")
def health_check():
    """Kiểm tra trạng thái sẵn sàng của mô hình"""
    if classifier.model is not None:
        return {
            "status": "healthy",
            "model_loaded": True,
            "device": classifier.device
        }
    return JSONResponse(
        status_code=503,
        content={"success": False, "error": "Hệ thống chưa sẵn sàng, lỗi tải mô hình."}
    )

@app.post("/generate")
async def generate_caption(file: UploadFile = File(...)):
    """Xử lý ảnh và sinh caption"""
    start_time = time.time()
    # Kiểm tra sự tồn tại của file
    if not file.filename:
        return JSONResponse(
            status_code=400,
            content={"success": False, "error": "Lỗi: Chưa đính kèm file ảnh."}
        )
    # Kiểm tra định dạng file
    if not file.content_type.startswith("image/"):
        return JSONResponse(
            status_code=400,
            content={"success": False, "error": "Invalid file format. Please upload a valid image (JPEG/PNG)."}
        )

    try:
        image_data = await file.read()
        # Kiểm tra tính toàn vẹn của ảnh
        try:
            image = Image.open(io.BytesIO(image_data)).convert("RGB")
        except UnidentifiedImageError:
            return JSONResponse(
                status_code=400,
                content={"success": False, "error": "Lỗi dữ liệu: File ảnh bị hỏng hoặc định dạng không được hỗ trợ."}
            )

        caption = classifier.predict(image)
        processing_time = round(time.time() - start_time, 2)

        return {
            "success": True,
            "filename": file.filename,
            "caption": caption,
            "processing_time_seconds": processing_time
        }

    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"success": False, "error": f"Lỗi hệ thống trong quá trình suy luận: {str(e)}"}
        )
# Khởi động Server trên Colab
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Chạy server ở một luồng riêng biệt để không làm treo Colab
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server FastAPI đã khởi chạy ngầm thành công tại http://0.0.0.0:8000")

### Call Local API

### ENDPOINT GET /

In [ ]:
import requests
API_URL = "http://127.0.0.1:8000/"
response = requests.get(API_URL)
print("Mã trạng thái HTTP:", response.status_code)
print("Kết quả trả về):")
print(response.json())

### ENDPOINT GET /health

In [ ]:
import requests
API_URL = "http://127.0.0.1:8000/health"
response = requests.get(API_URL)
print("Mã trạng thái HTTP:", response.status_code)
print("Kết quả trả về):")
print(response.json())

###ENDPOINT POST /generate

In [ ]:
import requests
img_url = 'https://thumbs.dreamstime.com/b/portrait-man-tiger-red-haired-white-shirt-lies-next-to-bengal-strokes-striped-belly-164248427.jpg'
sample_image_path = "sample_test.jpg"

with open(sample_image_path, 'wb') as f:
    f.write(requests.get(img_url).content)
print(f"Đã chuẩn bị xong ảnh mẫu: {sample_image_path}")

API_URL = "http://127.0.0.1:8000/generate"

print(f"\nĐang gửi request (POST) mang theo ảnh đến {API_URL} ...")

try:
    # Mở file ảnh ở chế độ đọc nhị phân (rb - read binary)
    with open(sample_image_path, "rb") as image_file:
        # Đóng gói file vào dictionary 'files'
        files = {"file": (sample_image_path, image_file, "image/jpeg")}
        response = requests.post(API_URL, files=files)

    print("Mã trạng thái HTTP:", response.status_code)
    print("Kết quả trả về:")
    print(response.json())

except Exception as e:
    print("Có lỗi xảy ra trong quá trình gọi API:", str(e))

### Call Public API


In [ ]:
# Mở luồng kết nối từ cổng 8000 của Colab ra Pinggy, chạy trên terminal của colab, nhấp phải chuột rồi chọn copy đường dẫn vì Ctrl + C sẽ ngắt kết nối
# ssh -p 443 -o StrictHostKeyChecking=no -R 80:localhost:8000 a.pinggy.io

In [ ]:
import requests

# 1. Nhập đường link Pinggy vào đây (theo cấu trúc link + /generate)
PUBLIC_API_URL = "http://ruajd-35-197-139-179.run.pinggy-free.link/generate"

sample_image_path = "sample_test.jpg"

print(f"Đang gửi ảnh đến máy chủ Public: {PUBLIC_API_URL} ...")

try:
    # Mở và đóng gói file ảnh
    with open(sample_image_path, "rb") as image_file:
        files = {"file": (sample_image_path, image_file, "image/jpeg")}
        response = requests.post(PUBLIC_API_URL, files=files)

    print("Mã trạng thái HTTP:", response.status_code)
    print("Kết quả trả về (JSON):")
    print(response.json())

except Exception as e:
    print("Có lỗi xảy ra trong quá trình gọi API Public:", str(e))